# 01. Sepsis Cohort (패혈증 코호트 정의)

## 목적
Sepsis-3 기준으로 패혈증 환자 코호트를 정의

-> src (파라미터 별도 설정) 없이도 실행 가능 버전

## Sepsis-3 정의
**감염 의심 (Suspected Infection)** + **SOFA ≥ 2점**

### 감염 의심 판정
- 항생제 투여 + 미생물 배양 검사가 ±24h 이내 동시 발생
- 둘 중 먼저 발생한 시점 = `suspected_infection_time`
- 항생제만 있는 경우(경험적 투여)도 포함

### SOFA 변화
- 감염 의심 시점 기준 최고 SOFA ≥ 2 (절대값 기준)

## 포함 기준
1. 성인 (18세 이상)
2. 첫 번째 ICU 입실
3. ICU 체류 ≥ 24시간
4. Sepsis-3 기준 충족

## 실행 요구사항
- **DuckDB 파일**: `mimic_total.duckdb` (MIMIC-IV 전체 테이블)
- **의존성**: `pandas`, `duckdb`만 필요
- **src/ 폴더 불필요**: 모든 코드가 이 노트북에 포함됨

## 출력
- `sepsis_cohort.csv`: 패혈증 환자 코호트 (1 row = 1 patient)

---
## 📌 설정값 (필요시 수정)

경로와 파라미터를 한곳에 모음. 다른 환경에서 실행 시 여기만 수정하면 됩니다.

In [19]:
from pathlib import Path
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
# ========================================
# 경로 설정 (환경에 맞게 수정)
# ========================================
DB_PATH = RAW_DIR / "mimic_total.duckdb"  # DuckDB 파일 경로
OUTPUT_PATH = PROCESSED_DIR / "sepsis_cohort.csv"  # 출력 CSV 경로

# ========================================
# 코호트 파라미터
# ========================================
MIN_AGE = 18                    # 최소 나이 (Sepsis-3 권고)
MIN_LOS_DAYS = 1.0              # 최소 ICU 체류 (24시간)
SOFA_THRESHOLD = 2              # Sepsis-3 정의
INFECTION_WINDOW_H = 24         # 항생제-배양 시간차 허용

# ========================================
# MIMIC-IV Item IDs
# ========================================

# 항생제 (11종) - IV 투여
ANTIBIOTIC_ITEMS = [
    225798,  # Vancomycin
    225893,  # Piperacillin/Tazobactam (Zosyn)
    225842,  # Ampicillin
    225850,  # Cefazolin
    225853,  # Ceftazidime
    225899,  # Bactrim (SMX/TMP)
    225851,  # Cefepime
    225859,  # Ciprofloxacin
    225883,  # Meropenem
    225837,  # Acyclovir
    225847,  # Aztreonam
]

# 승압제 (4종)
VASOPRESSOR_ITEMS = [
    221906,  # Norepinephrine
    221289,  # Epinephrine
    222315,  # Vasopressin
    221662,  # Dopamine
]

# SOFA 계산용 Item IDs
ITEM_PAO2 = 50821          # PaO2 (ABGA)
ITEM_FIO2 = 223835         # FiO2
ITEM_PLATELETS = 51265     # Platelets
ITEM_BILIRUBIN = 50885     # Bilirubin
ITEM_ABP_MEAN = 220052     # Arterial BP Mean
ITEM_NBP_MEAN = 220181     # Non-invasive BP Mean
ITEM_NOREPINEPHRINE = 221906
ITEM_EPINEPHRINE = 221289
ITEM_DOPAMINE = 221662
ITEM_GCS_EYE = 220739
ITEM_GCS_VERBAL = 223900
ITEM_GCS_MOTOR = 223901
ITEM_CREATININE = 50912
ITEM_LACTATE = 50813
ITEM_VENT = 225792         # Invasive Mechanical Ventilation
ITEM_DNR = 223758          # Code Status

print("✓ 설정값 로드 완료")
print(f"  DB 경로: {DB_PATH}")
print(f"  출력 경로: {OUTPUT_PATH}")
print(f"  코호트 기준: 나이≥{MIN_AGE}, LOS≥{MIN_LOS_DAYS}일, SOFA≥{SOFA_THRESHOLD}")

✓ 설정값 로드 완료
  DB 경로: DATA/raw/mimic_total.duckdb
  출력 경로: DATA/processed/sepsis_cohort.csv
  코호트 기준: 나이≥18, LOS≥1.0일, SOFA≥2


---
## 🔧 유틸리티 함수

재사용 가능한 헬퍼 함수들

In [20]:
import pandas as pd
import duckdb
import os

def items_to_sql(items):
    """Item ID 리스트를 SQL IN 절로 변환
    
    Args:
        items: int 또는 str 리스트
    Returns:
        "'id1', 'id2', 'id3'" 형태 문자열
    """
    return ', '.join([f"'{item}'" for item in items])

def save_csv(df, path, show_info=True):
    """DataFrame을 CSV로 저장하고 정보 출력"""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    
    if show_info:
        file_size = os.path.getsize(path) / (1024 * 1024)
        print(f"\n✓ 저장 완료: {os.path.basename(path)}")
        print(f"  파일 크기: {file_size:.2f} MB")
        print(f"  행 수: {len(df):,}개")
        print(f"  컬럼 수: {len(df.columns)}개")
        print(f"  경로: {path}")

print("✓ 유틸리티 함수 정의 완료")

✓ 유틸리티 함수 정의 완료


---
## 🗄️ DuckDB 연결

In [21]:
# DB 연결
con = duckdb.connect(DB_PATH)
print(f"✓ DuckDB 연결: {os.path.basename(DB_PATH)}")
print("\n=== 01. Sepsis Cohort 생성 시작 ===")

✓ DuckDB 연결: mimic_total.duckdb

=== 01. Sepsis Cohort 생성 시작 ===


---
## Step 1: 기본 코호트

성인 / 첫 ICU 입실 / 24시간 이상 체류

In [22]:
print("Step 1: 기본 코호트 (성인 / 첫 ICU / LOS ≥ 24h)")

cohort_query = f"""
WITH ranked_stays AS (
    SELECT 
        i.subject_id, i.hadm_id, i.stay_id,
        CAST(i.intime AS TIMESTAMP)  as intime,
        CAST(i.outtime AS TIMESTAMP) as outtime,
        CAST(i.los AS DOUBLE)        as los,
        i.first_careunit, i.last_careunit,
        CAST(p.anchor_age AS INTEGER) as anchor_age,
        p.gender,
        CAST(p.dod AS TIMESTAMP)      as dod,
        CAST(a.admittime AS TIMESTAMP) as admittime,
        CAST(a.dischtime AS TIMESTAMP) as dischtime,
        CAST(a.deathtime AS TIMESTAMP) as deathtime,
        a.hospital_expire_flag,
        ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) as icu_seq
    FROM icustays i
    INNER JOIN patients p   ON i.subject_id = p.subject_id
    INNER JOIN admissions a ON i.hadm_id    = a.hadm_id
    WHERE CAST(p.anchor_age AS INTEGER) >= {MIN_AGE}
      AND CAST(i.los AS DOUBLE)         >= {MIN_LOS_DAYS}
)
SELECT *,
    CASE WHEN deathtime IS NOT NULL AND deathtime <= outtime THEN 1 ELSE 0 END as icu_mortality,
    CASE WHEN hospital_expire_flag = '1' THEN 1 ELSE 0 END as hospital_mortality
FROM ranked_stays
WHERE icu_seq = 1
ORDER BY stay_id
"""

df_cohort = con.execute(cohort_query).df()
df_cohort = df_cohort.drop(columns=['icu_seq'])

print(f"✓ 기본 코호트: {len(df_cohort):,}명")
print(f"  ICU 사망: {df_cohort['icu_mortality'].sum():,}명 ({df_cohort['icu_mortality'].mean()*100:.1f}%)")
print(f"  병원 사망: {df_cohort['hospital_mortality'].sum():,}명 ({df_cohort['hospital_mortality'].mean()*100:.1f}%)")

# StringDtype → object 변환 (DuckDB 호환)
df_cohort = df_cohort.astype({c: 'object' for c in df_cohort.select_dtypes('string').columns})

# DuckDB에 등록 (다음 쿼리에서 사용)
con.register('cohort_df', df_cohort)
print(f"  → 'cohort_df' 등록 ({len(df_cohort):,} rows)")

Step 1: 기본 코호트 (성인 / 첫 ICU / LOS ≥ 24h)
✓ 기본 코호트: 54,551명
  ICU 사망: 3,837명 (7.0%)
  병원 사망: 5,881명 (10.8%)
  → 'cohort_df' 등록 (54,551 rows)


---
## Step 2: 감염 의심 시간 (Suspected Infection Time)

### 로직
1. `inputevents` → IV 항생제 최초 투여 시점 (`abx_start`)
2. `microbiologyevents` → 배양 최초 채취 시점 (`culture_time`)
3. 판정:
   - 둘 다 있고 ±24h 이내 → `min(abx_start, culture_time)`
   - 항생제만 있음 → `abx_start` (경험적 투여)
   - 배양만 있음 → 제외 (항생제 없으면 감염 의심 아님)

In [23]:
print("\nStep 2: 감염 의심 시간 추출")

abx_items_sql = items_to_sql(ANTIBIOTIC_ITEMS)
infection_window_sec = INFECTION_WINDOW_H * 3600

infection_query = f"""
WITH abx AS (
    SELECT c.stay_id, c.subject_id, c.hadm_id,
           MIN(CAST(ie.starttime AS TIMESTAMP)) as abx_start
    FROM cohort_df c
    INNER JOIN inputevents ie ON c.stay_id = ie.stay_id
    WHERE ie.itemid IN ({abx_items_sql})
      AND ie.starttime IS NOT NULL
    GROUP BY c.stay_id, c.subject_id, c.hadm_id
),
culture AS (
    SELECT c.stay_id, c.subject_id,
           MIN(CAST(me.charttime AS TIMESTAMP)) as culture_time
    FROM cohort_df c
    INNER JOIN microbiologyevents me
        ON c.subject_id = me.subject_id AND c.hadm_id = me.hadm_id
    WHERE me.charttime IS NOT NULL
    GROUP BY c.stay_id, c.subject_id
)
SELECT
    a.stay_id,
    a.abx_start,
    cu.culture_time,
    CASE
        WHEN a.abx_start IS NOT NULL AND cu.culture_time IS NOT NULL
             AND ABS(EXTRACT(EPOCH FROM (a.abx_start - cu.culture_time))) <= {infection_window_sec}
        THEN LEAST(a.abx_start, cu.culture_time)
        WHEN a.abx_start IS NOT NULL AND cu.culture_time IS NULL
        THEN a.abx_start
        ELSE NULL
    END as suspected_infection_time,
    CASE
        WHEN a.abx_start IS NOT NULL AND cu.culture_time IS NOT NULL
             AND ABS(EXTRACT(EPOCH FROM (a.abx_start - cu.culture_time))) <= {infection_window_sec}
        THEN 1 ELSE 0
    END as abx_culture_both
FROM abx a
LEFT JOIN culture cu ON a.stay_id = cu.stay_id
"""

df_infection = con.execute(infection_query).df()

print(f"✓ 항생제 투여: {len(df_infection):,}명")
print(f"  감염 의심 판정: {df_infection['suspected_infection_time'].notna().sum():,}명")
print(f"  항생제+배양 동시: {df_infection['abx_culture_both'].sum():,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(
    df_infection[['stay_id', 'suspected_infection_time', 'abx_culture_both']],
    on='stay_id',
    how='left'
)

n_no_infection = df_cohort['suspected_infection_time'].isna().sum()
print(f"  감염 의심 없음: {n_no_infection:,}명 (이후 제외)")

# 감염 의심 있는 환자만
df_cohort = df_cohort[df_cohort['suspected_infection_time'].notna()].copy()

# StringDtype → object 변환 (DuckDB 호환)
df_cohort = df_cohort.astype({c: 'object' for c in df_cohort.select_dtypes('string').columns})

con.register('cohort_df', df_cohort)
print(f"  → cohort_df 업데이트 ({len(df_cohort):,} rows)")


Step 2: 감염 의심 시간 추출
✓ 항생제 투여: 29,478명
  감염 의심 판정: 18,842명
  항생제+배양 동시: 16,151명
  감염 의심 없음: 35,709명 (이후 제외)
  → cohort_df 업데이트 (18,842 rows)


---
## Step 3: SOFA Score 계산

감염 의심 시점 기준 ±48h 내 최고 SOFA 점수 계산

### SOFA 6개 장기
1. **Respiration(호흡)**: PaO2/FiO2 ratio (동맥혈 산소분압/흡입산소 농도)
2. **Coagulation(응고)**: Platelets (혈소판 수치)
3. **Liver(간)**: Bilirubin 
4. **Cardiovascular(심혈관)**: MAP (평균 동맥압) or Vasopressor (씅압제 사용 여부)
5. **CNS(중추신경계)**: GCS (의식 수준 평가)
6. **Renal(신장/콩팥)**: Creatinine (신장 기능 지표)

In [24]:
print("\nStep 3: SOFA Score 계산 (감염시점 ±48h 내 최고값)")
print(f"  대상: {len(df_cohort):,}명")

vaso_items_sql = items_to_sql(VASOPRESSOR_ITEMS)

sofa_query = f"""
WITH tw AS (
    SELECT stay_id, subject_id, hadm_id, intime,
           suspected_infection_time,
           suspected_infection_time - INTERVAL '48' HOUR as ws,
           suspected_infection_time + INTERVAL '48' HOUR as we
    FROM cohort_df
),

-- (1) 호흡: PaO2/FiO2
pao2_d AS (
    SELECT tw.stay_id, TRY_CAST(l.valuenum AS DOUBLE) as pao2
    FROM tw INNER JOIN labevents l ON tw.subject_id = l.subject_id
    WHERE l.itemid = '{ITEM_PAO2}'
      AND CAST(l.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(l.valuenum AS DOUBLE) > 0
),
fio2_d AS (
    SELECT tw.stay_id, TRY_CAST(ce.valuenum AS DOUBLE) / 100.0 as fio2
    FROM tw INNER JOIN chartevents ce ON tw.stay_id = ce.stay_id
    WHERE ce.itemid = '{ITEM_FIO2}'
      AND CAST(ce.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(ce.valuenum AS DOUBLE) > 0
),
resp AS (
    SELECT p.stay_id,
           MIN(p.pao2 / COALESCE(NULLIF(f.fio2, 0), 0.21)) as min_pf
    FROM pao2_d p LEFT JOIN fio2_d f ON p.stay_id = f.stay_id
    GROUP BY p.stay_id
),

-- (2) 응고: Platelets
plt AS (
    SELECT tw.stay_id, MIN(TRY_CAST(l.valuenum AS DOUBLE)) as min_plt
    FROM tw INNER JOIN labevents l ON tw.subject_id = l.subject_id
    WHERE l.itemid = '{ITEM_PLATELETS}'
      AND CAST(l.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(l.valuenum AS DOUBLE) > 0
    GROUP BY tw.stay_id
),

-- (3) 간: Bilirubin
bili AS (
    SELECT tw.stay_id, MAX(TRY_CAST(l.valuenum AS DOUBLE)) as max_bili
    FROM tw INNER JOIN labevents l ON tw.subject_id = l.subject_id
    WHERE l.itemid = '{ITEM_BILIRUBIN}'
      AND CAST(l.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(l.valuenum AS DOUBLE) >= 0
    GROUP BY tw.stay_id
),

-- (4) 순환: MAP
map_d AS (
    SELECT tw.stay_id, MIN(TRY_CAST(ce.valuenum AS DOUBLE)) as min_map
    FROM tw INNER JOIN chartevents ce ON tw.stay_id = ce.stay_id
    WHERE ce.itemid IN ('{ITEM_ABP_MEAN}', '{ITEM_NBP_MEAN}')
      AND CAST(ce.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(ce.valuenum AS DOUBLE) > 0
    GROUP BY tw.stay_id
),
-- (4b) 순환: 승압제 용량
vaso AS (
    SELECT tw.stay_id,
           MAX(CASE WHEN ie.itemid='{ITEM_DOPAMINE}'       THEN TRY_CAST(ie.rate AS DOUBLE) ELSE 0 END) as dopa,
           MAX(CASE WHEN ie.itemid='{ITEM_NOREPINEPHRINE}' THEN TRY_CAST(ie.rate AS DOUBLE) ELSE 0 END) as norepi,
           MAX(CASE WHEN ie.itemid='{ITEM_EPINEPHRINE}'    THEN TRY_CAST(ie.rate AS DOUBLE) ELSE 0 END) as epi
    FROM tw INNER JOIN inputevents ie ON tw.stay_id = ie.stay_id
    WHERE ie.itemid IN ({vaso_items_sql})
      AND CAST(ie.starttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(ie.rate AS DOUBLE) > 0
    GROUP BY tw.stay_id
),

-- (5) 신경: GCS
gcs AS (
    SELECT tw.stay_id,
           COALESCE(MIN(CASE WHEN ce.itemid='{ITEM_GCS_EYE}'    THEN TRY_CAST(ce.valuenum AS DOUBLE) END), 4) +
           COALESCE(MIN(CASE WHEN ce.itemid='{ITEM_GCS_VERBAL}' THEN TRY_CAST(ce.valuenum AS DOUBLE) END), 5) +
           COALESCE(MIN(CASE WHEN ce.itemid='{ITEM_GCS_MOTOR}'  THEN TRY_CAST(ce.valuenum AS DOUBLE) END), 6)
           as min_gcs
    FROM tw INNER JOIN chartevents ce ON tw.stay_id = ce.stay_id
    WHERE ce.itemid IN ('{ITEM_GCS_EYE}', '{ITEM_GCS_VERBAL}', '{ITEM_GCS_MOTOR}')
      AND CAST(ce.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(ce.valuenum AS DOUBLE) IS NOT NULL
    GROUP BY tw.stay_id
),

-- (6) 신장: Creatinine
cr AS (
    SELECT tw.stay_id, MAX(TRY_CAST(l.valuenum AS DOUBLE)) as max_cr
    FROM tw INNER JOIN labevents l ON tw.subject_id = l.subject_id
    WHERE l.itemid = '{ITEM_CREATININE}'
      AND CAST(l.charttime AS TIMESTAMP) BETWEEN tw.ws AND tw.we
      AND TRY_CAST(l.valuenum AS DOUBLE) > 0
    GROUP BY tw.stay_id
),

-- SOFA 점수 변환
scored AS (
    SELECT tw.stay_id,
        CASE WHEN r.min_pf  < 100 THEN 4 WHEN r.min_pf  < 200 THEN 3
             WHEN r.min_pf  < 300 THEN 2 WHEN r.min_pf  < 400 THEN 1 ELSE 0 END as sofa_resp,
        CASE WHEN p.min_plt < 20  THEN 4 WHEN p.min_plt < 50  THEN 3
             WHEN p.min_plt < 100 THEN 2 WHEN p.min_plt < 150 THEN 1 ELSE 0 END as sofa_coag,
        CASE WHEN b.max_bili >= 12 THEN 4 WHEN b.max_bili >= 6  THEN 3
             WHEN b.max_bili >= 2  THEN 2 WHEN b.max_bili >= 1.2 THEN 1 ELSE 0 END as sofa_liver,
        CASE WHEN COALESCE(v.dopa,0)>15  OR COALESCE(v.epi,0)>0.1  OR COALESCE(v.norepi,0)>0.1  THEN 4
             WHEN COALESCE(v.dopa,0)>5   OR COALESCE(v.epi,0)>0    OR COALESCE(v.norepi,0)>0    THEN 3
             WHEN COALESCE(v.dopa,0)>0   THEN 2
             WHEN m.min_map < 70          THEN 1 ELSE 0 END as sofa_cardio,
        CASE WHEN g.min_gcs < 6  THEN 4 WHEN g.min_gcs < 10 THEN 3
             WHEN g.min_gcs < 13 THEN 2 WHEN g.min_gcs < 15 THEN 1 ELSE 0 END as sofa_cns,
        CASE WHEN cr.max_cr >= 5.0 THEN 4 WHEN cr.max_cr >= 3.5 THEN 3
             WHEN cr.max_cr >= 2.0 THEN 2 WHEN cr.max_cr >= 1.2 THEN 1 ELSE 0 END as sofa_renal
    FROM tw
    LEFT JOIN resp  r  ON tw.stay_id = r.stay_id
    LEFT JOIN plt   p  ON tw.stay_id = p.stay_id
    LEFT JOIN bili  b  ON tw.stay_id = b.stay_id
    LEFT JOIN map_d m  ON tw.stay_id = m.stay_id
    LEFT JOIN vaso  v  ON tw.stay_id = v.stay_id
    LEFT JOIN gcs   g  ON tw.stay_id = g.stay_id
    LEFT JOIN cr    cr ON tw.stay_id = cr.stay_id
)
SELECT *,
    (COALESCE(sofa_resp,0) + COALESCE(sofa_coag,0) + COALESCE(sofa_liver,0) +
     COALESCE(sofa_cardio,0) + COALESCE(sofa_cns,0) + COALESCE(sofa_renal,0)
    ) as sofa_total
FROM scored
"""

print("  쿼리 실행 중... (1-2분 소요)")
df_sofa = con.execute(sofa_query).df()

print(f"✓ SOFA 계산 완료: {len(df_sofa):,}명")
print(f"  평균: {df_sofa['sofa_total'].mean():.1f}, 중앙값: {df_sofa['sofa_total'].median():.1f}")
print(f"  SOFA ≥ {SOFA_THRESHOLD}: {(df_sofa['sofa_total'] >= SOFA_THRESHOLD).sum():,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_sofa, on='stay_id', how='left')
df_cohort['sofa_total'] = df_cohort['sofa_total'].fillna(0)


Step 3: SOFA Score 계산 (감염시점 ±48h 내 최고값)
  대상: 18,842명
  쿼리 실행 중... (1-2분 소요)
✓ SOFA 계산 완료: 18,842명
  평균: 9.0, 중앙값: 9.0
  SOFA ≥ 2: 18,001명


---
## Step 4: Sepsis-3 필터링

감염 의심 + SOFA ≥ 2

In [25]:
print("\nStep 4: Sepsis-3 기준 적용 (SOFA ≥ 2)")

n_before = len(df_cohort)
df_sepsis = df_cohort[df_cohort['sofa_total'] >= SOFA_THRESHOLD].copy()
n_after = len(df_sepsis)

print(f"✓ Sepsis 환자: {n_after:,}명 ({n_after/n_before*100:.1f}%)")
print(f"  제외: {n_before - n_after:,}명 (SOFA < 2)")

# StringDtype → object 변환 (DuckDB 호환)
df_sepsis = df_sepsis.astype({c: 'object' for c in df_sepsis.select_dtypes('string').columns})

# 다음 단계를 위해 등록
con.register('sepsis_df', df_sepsis)


Step 4: Sepsis-3 기준 적용 (SOFA ≥ 2)
✓ Sepsis 환자: 18,001명 (95.5%)
  제외: 841명 (SOFA < 2)


---
## Step 5: Septic Shock 판정

Sepsis + 승압제 + Lactate > 2 mmol/L

In [26]:
print("\nStep 5: Septic Shock 판정")

vaso_items_sql = items_to_sql(VASOPRESSOR_ITEMS)

# StringDtype → object 변환 (DuckDB 호환)
df_sepsis = df_sepsis.astype({c: 'object' for c in df_sepsis.select_dtypes('string').columns})

con.register('sepsis_df', df_sepsis)

shock_query = f"""
WITH vaso_t AS (
    SELECT s.stay_id,
           MIN(CAST(ie.starttime AS TIMESTAMP)) as first_vaso
    FROM sepsis_df s
    INNER JOIN inputevents ie ON s.stay_id = ie.stay_id
    WHERE ie.itemid IN ({vaso_items_sql})
      AND TRY_CAST(ie.rate AS DOUBLE) > 0
    GROUP BY s.stay_id
),
lac_high AS (
    SELECT s.stay_id,
           MIN(CAST(l.charttime AS TIMESTAMP)) as first_high_lac
    FROM sepsis_df s
    INNER JOIN labevents l ON s.subject_id = l.subject_id
    WHERE l.itemid = '{ITEM_LACTATE}'
      AND TRY_CAST(l.valuenum AS DOUBLE) > 2.0
      AND CAST(l.charttime AS TIMESTAMP) BETWEEN s.intime AND s.outtime
    GROUP BY s.stay_id
)
SELECT v.stay_id,
    CASE WHEN v.first_vaso IS NOT NULL AND lh.first_high_lac IS NOT NULL
              AND ABS(EXTRACT(EPOCH FROM (v.first_vaso - lh.first_high_lac))) <= {INFECTION_WINDOW_H * 3600}
         THEN GREATEST(v.first_vaso, lh.first_high_lac)
         ELSE NULL
    END as septic_shock_time
FROM vaso_t v
LEFT JOIN lac_high lh ON v.stay_id = lh.stay_id
"""

df_shock = con.execute(shock_query).df()

df_sepsis = df_sepsis.merge(df_shock, on='stay_id', how='left')
df_sepsis['septic_shock_flag'] = df_sepsis['septic_shock_time'].notna().astype(int)

print(f"✓ Septic Shock: {df_sepsis['septic_shock_flag'].sum():,}명 ({df_sepsis['septic_shock_flag'].mean()*100:.1f}%)")

# StringDtype → object 변환 (DuckDB 호환)
df_sepsis = df_sepsis.astype({c: 'object' for c in df_sepsis.select_dtypes('string').columns})

con.register('sepsis_df', df_sepsis)


Step 5: Septic Shock 판정
✓ Septic Shock: 3,756명 (20.9%)


---
## Step 6: 이벤트 시간 추출

DNR, Ventilation, Pressor 시작 시간

In [27]:
print("\nStep 6: 이벤트 시간 추출 (DNR / Vent / Pressor)")

# DNR
dnr_query = f"""
SELECT s.stay_id,
       MIN(CAST(ce.charttime AS TIMESTAMP)) as dnr_time
FROM sepsis_df s
INNER JOIN chartevents ce ON s.stay_id = ce.stay_id
WHERE ce.itemid = '{ITEM_DNR}'
GROUP BY s.stay_id
"""
df_dnr = con.execute(dnr_query).df()
print(f"  DNR: {len(df_dnr):,}명")

# Ventilation
vent_query = f"""
SELECT s.stay_id,
       MIN(CAST(pe.starttime AS TIMESTAMP)) as vent_start_time
FROM sepsis_df s
INNER JOIN procedureevents pe ON s.stay_id = pe.stay_id
WHERE pe.itemid = '{ITEM_VENT}'
  AND pe.starttime IS NOT NULL
GROUP BY s.stay_id
"""
df_vent = con.execute(vent_query).df()
print(f"  Vent: {len(df_vent):,}명")

# Pressor
pressor_query = f"""
SELECT s.stay_id,
       MIN(CAST(ie.starttime AS TIMESTAMP)) as pressor_start_time
FROM sepsis_df s
INNER JOIN inputevents ie ON s.stay_id = ie.stay_id
WHERE ie.itemid IN ({vaso_items_sql})
  AND ie.starttime IS NOT NULL
  AND CAST(ie.rate AS DOUBLE) > 0
GROUP BY s.stay_id
"""
df_pressor = con.execute(pressor_query).df()
print(f"  Pressor: {len(df_pressor):,}명")

# 병합
df_sepsis = df_sepsis.merge(df_dnr, on='stay_id', how='left')
df_sepsis = df_sepsis.merge(df_vent, on='stay_id', how='left')
df_sepsis = df_sepsis.merge(df_pressor, on='stay_id', how='left')

print("✓ 이벤트 시간 병합 완료")


Step 6: 이벤트 시간 추출 (DNR / Vent / Pressor)
  DNR: 9,696명
  Vent: 11,426명
  Pressor: 6,556명
✓ 이벤트 시간 병합 완료


---
## Step 7: 결과 요약

In [28]:
print("\n" + "="*60)
print("Sepsis Cohort 요약")
print("="*60)

# Step 1의 n_total 계산
con_temp = duckdb.connect(DB_PATH)
n_total = con_temp.execute(f"""
    SELECT COUNT(DISTINCT i.stay_id)
    FROM icustays i
    INNER JOIN patients p ON i.subject_id = p.subject_id
    WHERE CAST(p.anchor_age AS INTEGER) >= {MIN_AGE}
      AND CAST(i.los AS DOUBLE) >= {MIN_LOS_DAYS}
""").fetchone()[0]
con_temp.close()

n_sepsis = len(df_sepsis)

print(f"\n총 환자 수: {n_sepsis:,}명 (전체 {n_total:,}명 중 {n_sepsis/n_total*100:.1f}%)")

print(f"\n=== 사망률 ===")
print(f"  ICU 사망: {df_sepsis['icu_mortality'].sum():,}명 ({df_sepsis['icu_mortality'].mean()*100:.1f}%)")
print(f"  병원 사망: {df_sepsis['hospital_mortality'].sum():,}명 ({df_sepsis['hospital_mortality'].mean()*100:.1f}%)")

print(f"\n=== Septic Shock ===")
print(f"  Septic Shock: {df_sepsis['septic_shock_flag'].sum():,}명 ({df_sepsis['septic_shock_flag'].mean()*100:.1f}% of Sepsis)")

print(f"\n=== SOFA 분포 ===")
print(f"  평균: {df_sepsis['sofa_total'].mean():.1f}")
print(f"  중앙값: {df_sepsis['sofa_total'].median():.0f}")
print(f"  범위: {df_sepsis['sofa_total'].min():.0f} ~ {df_sepsis['sofa_total'].max():.0f}")

print(f"\n=== 이벤트 발생 환자 ===")
print(f"  DNR: {df_sepsis['dnr_time'].notna().sum():,}명 ({df_sepsis['dnr_time'].notna().mean()*100:.1f}%)")
print(f"  Ventilation: {df_sepsis['vent_start_time'].notna().sum():,}명 ({df_sepsis['vent_start_time'].notna().mean()*100:.1f}%)")
print(f"  Pressor: {df_sepsis['pressor_start_time'].notna().sum():,}명 ({df_sepsis['pressor_start_time'].notna().mean()*100:.1f}%)")

print(f"\n=== 컬럼 목록 ({len(df_sepsis.columns)}개) ===")
for col in df_sepsis.columns:
    print(f"  - {col}")


Sepsis Cohort 요약



총 환자 수: 18,001명 (전체 74,829명 중 24.1%)

=== 사망률 ===
  ICU 사망: 1,787명 (9.9%)
  병원 사망: 2,484명 (13.8%)

=== Septic Shock ===
  Septic Shock: 3,756명 (20.9% of Sepsis)

=== SOFA 분포 ===
  평균: 9.3
  중앙값: 9
  범위: 2 ~ 23

=== 이벤트 발생 환자 ===
  DNR: 9,696명 (53.9%)
  Ventilation: 11,426명 (63.5%)
  Pressor: 6,556명 (36.4%)

=== 컬럼 목록 (31개) ===
  - subject_id
  - hadm_id
  - stay_id
  - intime
  - outtime
  - los
  - first_careunit
  - last_careunit
  - anchor_age
  - gender
  - dod
  - admittime
  - dischtime
  - deathtime
  - hospital_expire_flag
  - icu_mortality
  - hospital_mortality
  - suspected_infection_time
  - abx_culture_both
  - sofa_resp
  - sofa_coag
  - sofa_liver
  - sofa_cardio
  - sofa_cns
  - sofa_renal
  - sofa_total
  - septic_shock_time
  - septic_shock_flag
  - dnr_time
  - vent_start_time
  - pressor_start_time


---
## Step 8: 저장

In [29]:
save_csv(df_sepsis, OUTPUT_PATH)

print("\n=== 저장된 샘플 데이터 ===")
df_sepsis.head()


✓ 저장 완료: sepsis_cohort.csv
  파일 크기: 5.11 MB
  행 수: 18,001개
  컬럼 수: 31개
  경로: DATA/processed/sepsis_cohort.csv

=== 저장된 샘플 데이터 ===


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit,last_careunit,anchor_age,gender,...,sofa_liver,sofa_cardio,sofa_cns,sofa_renal,sofa_total,septic_shock_time,septic_shock_flag,dnr_time,vent_start_time,pressor_start_time
0,12207593,22795209,30000646,2194-04-29 01:39:22,2194-05-03 18:23:48,4.697523,Coronary Care Unit (CCU),Coronary Care Unit (CCU),43,M,...,0,1,0,0,5,NaT,0,2194-04-29 01:49:00,NaT,NaT
1,16513856,24463832,30001446,2186-04-12 03:49:00,2186-04-13 19:45:47,1.664433,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),56,M,...,3,3,1,2,12,NaT,0,2186-04-12 04:21:00,NaT,2186-04-12 05:36:00
2,11440929,23071615,30003749,2180-06-06 16:03:00,2180-06-08 08:19:44,1.678287,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),50,M,...,2,4,4,4,20,2180-06-06 16:35:00,1,NaT,2180-06-06 16:30:00,2180-06-06 16:30:00
3,17220323,25700666,30004242,2180-11-19 01:06:36,2180-11-20 20:53:44,1.824398,Trauma SICU (TSICU),Trauma SICU (TSICU),75,F,...,0,1,2,0,3,NaT,0,2180-11-20 00:14:00,NaT,NaT
4,11546816,27336674,30005474,2151-08-03 18:46:39,2151-08-05 16:02:52,1.886262,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),91,M,...,0,1,1,1,5,NaT,0,NaT,NaT,NaT


In [30]:
con.close()
print("\n=== 01. Sepsis Cohort 생성 완료 ===")


=== 01. Sepsis Cohort 생성 완료 ===


---
## +a: 검증

In [ ]:
# 코호트 비율 (기대: 30-50%)
print("=== 코호트 비율 검증 ===")
ratio = len(df_sepsis) / n_total * 100
print(f"  전체 대비 Sepsis: {ratio:.1f}%")
if ratio < 20:
    print("  ⚠️ 낮음 - 항생제 itemid 또는 배양 조건 확인")
elif ratio > 60:
    print("  ⚠️ 높음 - SOFA 계산 로직 확인")

=== 코호트 비율 검증 ===
  전체 대비 Sepsis: 24.1% (기대 30-50%)


In [32]:
# 감염 의심 시간 분포
print("\n=== 감염 의심 시간 (ICU 입실 대비) ===")
df_sepsis['hours_to_infection'] = (
    (pd.to_datetime(df_sepsis['suspected_infection_time']) -
     pd.to_datetime(df_sepsis['intime']))
    .dt.total_seconds() / 3600
)
print(f"  평균: {df_sepsis['hours_to_infection'].mean():.1f}h")
print(f"  중앙값: {df_sepsis['hours_to_infection'].median():.1f}h")
print(f"  입실 전 감염 의심: {(df_sepsis['hours_to_infection'] < 0).sum():,}명")
print(f"  입실 후 24h 이내: {((df_sepsis['hours_to_infection'] >= 0) & (df_sepsis['hours_to_infection'] <= 24)).sum():,}명")


=== 감염 의심 시간 (ICU 입실 대비) ===
  평균: 4.6h
  중앙값: 2.2h
  입실 전 감염 의심: 2,031명
  입실 후 24h 이내: 15,281명


In [33]:
# Sepsis vs Non-Sepsis 사망률 비교
print("\n=== Sepsis vs Non-Sepsis ===")

# 전체 코호트 다시 로드 (비교용)
con_temp = duckdb.connect(DB_PATH)
all_cohort = con_temp.execute(f"""
WITH ranked_stays AS (
    SELECT 
        i.stay_id,
        CAST(a.deathtime AS TIMESTAMP) as deathtime,
        CAST(i.outtime AS TIMESTAMP) as outtime,
        a.hospital_expire_flag,
        ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) as icu_seq
    FROM icustays i
    INNER JOIN patients p   ON i.subject_id = p.subject_id
    INNER JOIN admissions a ON i.hadm_id    = a.hadm_id
    WHERE CAST(p.anchor_age AS INTEGER) >= {MIN_AGE}
      AND CAST(i.los AS DOUBLE) >= {MIN_LOS_DAYS}
)
SELECT
    stay_id,
    CASE WHEN deathtime IS NOT NULL AND deathtime <= outtime THEN 1 ELSE 0 END as icu_mortality,
    CASE WHEN hospital_expire_flag = '1' THEN 1 ELSE 0 END as hospital_mortality
FROM ranked_stays
WHERE icu_seq = 1
""").df()
con_temp.close()

non_sepsis = all_cohort[~all_cohort['stay_id'].isin(df_sepsis['stay_id'])]

print(f"  Sepsis  ICU 사망률: {df_sepsis['icu_mortality'].mean()*100:.1f}%  |  병원: {df_sepsis['hospital_mortality'].mean()*100:.1f}%")
print(f"  Non-Sep ICU 사망률: {non_sepsis['icu_mortality'].mean()*100:.1f}%  |  병원: {non_sepsis['hospital_mortality'].mean()*100:.1f}%")


=== Sepsis vs Non-Sepsis ===
  Sepsis  ICU 사망률: 9.9%  |  병원: 13.8%
  Non-Sep ICU 사망률: 5.6%  |  병원: 9.3%


---
## +b: Sepsis 비율 시각화

코호트 선정 흐름 / 구성 비율 / 사망률 비교 / SOFA 분포

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

# ── CSV 로드 ──────────────────────────────────────────────────
df = pd.read_csv(
    Path('../data/processed/sepsis_cohort.csv'),
    parse_dates=['intime', 'outtime', 'deathtime', 'dnr_time',
                 'vent_start_time', 'pressor_start_time']
)
print(f'✓ 로드 완료: {len(df):,}명')

# ── 색상 팔레트 ──────────────────────────────────────────────
C_SEPSIS = '#FFA726'
C_SHOCK  = '#EF5350'
C_BLUE   = '#5C6BC0'

# ── 수치 계산 ────────────────────────────────────────────────
n_sep      = len(df)
n_shock    = int(df['septic_shock_flag'].sum())
n_sep_only = n_sep - n_shock

df_shock = df[df['septic_shock_flag'] == 1]
df_nosho = df[df['septic_shock_flag'] == 0]

sofa_all   = df['sofa_total'].dropna().astype(int)
sofa_shock = df_shock['sofa_total'].dropna().astype(int)
sofa_nosho = df_nosho['sofa_total'].dropna().astype(int)

pct_dnr   = df['dnr_time'].notna().mean() * 100
pct_vent  = df['vent_start_time'].notna().mean() * 100
pct_press = df['pressor_start_time'].notna().mean() * 100

# ── Figure ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sepsis Cohort Overview', fontsize=15, fontweight='bold')

# ── Plot 1: Donut — Sepsis vs Septic Shock 구성 ──────────────
ax1 = axes[0, 0]
wedges, texts = ax1.pie(
    [n_sep_only, n_shock],
    labels=[
        f'Sepsis (without Shock)\n{n_sep_only:,}명 ({n_sep_only/n_sep*100:.1f}%)',
        f'Septic Shock\n{n_shock:,}명 ({n_shock/n_sep*100:.1f}%)',
    ],
    colors=[C_SEPSIS, C_SHOCK],
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white'),
)
for t in texts:
    t.set_fontsize(9)
ax1.text(0, 0, f'{n_sep:,}명\n총 Sepsis', ha='center', va='center',
         fontsize=10, fontweight='bold')
ax1.set_title('Sepsis 코호트 구성', fontsize=11, fontweight='bold')

# ── Plot 2: 사망률 — Septic Shock vs Sepsis only ─────────────
ax2 = axes[0, 1]
x = np.arange(2)
w = 0.32
shock_mort = [df_shock['icu_mortality'].mean()*100, df_shock['hospital_mortality'].mean()*100]
nosho_mort = [df_nosho['icu_mortality'].mean()*100, df_nosho['hospital_mortality'].mean()*100]
b1 = ax2.bar(x - w/2, shock_mort, w, label='Septic Shock', color=C_SHOCK,  alpha=0.88, edgecolor='white')
b2 = ax2.bar(x + w/2, nosho_mort, w, label='Sepsis only',  color=C_SEPSIS, alpha=0.88, edgecolor='white')
for bar, val in zip(b1, shock_mort):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, val in zip(b2, nosho_mort):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(['ICU 사망률', '병원 사망률'], fontsize=10)
ax2.set_ylabel('사망률 (%)', fontsize=9)
ax2.set_ylim(0, max(shock_mort + nosho_mort) * 1.3)
ax2.set_title('Septic Shock vs Sepsis only 사망률', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

# ── Plot 3: SOFA 분포 (Shock 여부 구분) ──────────────────────
ax3 = axes[1, 0]
bins = range(int(sofa_all.min()), int(sofa_all.max()) + 2)
ax3.hist(sofa_nosho, bins=bins, color=C_SEPSIS, alpha=0.75,
         edgecolor='white', align='left', label='Sepsis only')
ax3.hist(sofa_shock, bins=bins, color=C_SHOCK,  alpha=0.75,
         edgecolor='white', align='left', label='Septic Shock')
ax3.axvline(sofa_all.median(), color=C_BLUE, linestyle='--', linewidth=1.8,
            label=f'전체 중앙값: {sofa_all.median():.0f}')
ax3.set_xlabel('SOFA Score', fontsize=9)
ax3.set_ylabel('환자 수', fontsize=9)
ax3.set_title('SOFA Score 분포 (Shock 여부)', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
ax3.spines[['top', 'right']].set_visible(False)

# ── Plot 4: 임상 이벤트 발생률 ────────────────────────────────
ax4 = axes[1, 1]
events = ['DNR', 'Ventilation', 'Pressor']
pcts   = [pct_dnr, pct_vent, pct_press]
colors = ['#78909C', '#42A5F5', '#AB47BC']
bars = ax4.bar(events, pcts, color=colors, alpha=0.88, edgecolor='white')
for bar, val in zip(bars, pcts):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax4.set_ylabel('발생률 (%)', fontsize=9)
ax4.set_ylim(0, 100)
ax4.set_title('주요 임상 이벤트 발생률\n(Sepsis 코호트 내)', fontsize=11, fontweight='bold')
ax4.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/sepsis_cohort_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 저장 완료: DATA/processed/sepsis_cohort_overview.png')
